%md
# 📑 Other Monthly Billing for Finance Report
 
## 🔗 Links
- **[BRD - Orbit / Light / Pearl / Cell / Euclid / Monza Monthly Billing for Finance Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/239730701/BRD+-+Orbit+Light+Pearl+Cell+Euclid+Monza+Monthly+Billing+for+Finance+Report)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-345)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-960)**  
  A summary of findings from UAT.
 
---

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')
#env_prefix = 'p'

## Input Parameters

**report_args:** it is a json which contains all the appropriate parameters to run the report

`{
   "portfolio_name": ["Orbit"],
   "output_file_prefix": "Orbit_MonthlyBilling_",
   "report_output": "/Volumes/qa-cdh-delivery/internalreports/monthlybilling/orbit/",
   "ad_hoc_date" : "YYYY-MM-DD"
}`


**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
from datetime import datetime
from pytz import timezone
import json

athens_tz = timezone('Europe/Athens')

# --- Get report_args and report_name ---
try:
    report_args = dbutils.widgets.get("report_args").replace("'", '"')
except Exception:
    raise ValueError("Missing required input: report_args")

try:
    report_name = dbutils.widgets.get("report_name")
except Exception:
    raise ValueError("Missing required input: report_name")

# --- Optional run_date widget ---
try:
    run_date_widget = dbutils.widgets.get("run_date")
except:
    run_date_widget = ""

# --- Parse JSON ---
json_data = json.loads(report_args)

required_keys = ["portfolio_name", "output_file_prefix", "report_output"]
for key in required_keys:
    if key not in json_data:
        raise KeyError(f"'{key}' key is missing in the report_args JSON data.")

portfolioName = json_data["portfolio_name"]
output_file_prefix = json_data["output_file_prefix"]
report_output = json_data["report_output"]

# --- Resolve ad_hoc_date vs run_date vs today ---
if "ad_hoc_date" in json_data and json_data["ad_hoc_date"]:
    try:
        ad_hoc_date = datetime.strptime(json_data["ad_hoc_date"], '%Y-%m-%d').date()
        eom_date = ad_hoc_date
    except Exception:
        raise ValueError(f"Invalid ad_hoc_date format: {json_data['ad_hoc_date']} — use YYYY-MM-DD")
else:
    if run_date_widget:
        try:
            run_date = datetime.strptime(run_date_widget, '%Y-%m-%d').date()
        except Exception:
            raise ValueError(f"Invalid run_date format: {run_date_widget} — use YYYY-MM-DD")
    else:
        run_date = datetime.now(athens_tz).date()

    # Use Spark SQL to calculate EOM date (end of previous month) unless an ad_hoc_date is given then eom_date = ad_hoc_date
    spark.sql(f"CREATE OR REPLACE TEMP VIEW tmp_run_date AS SELECT DATE('{run_date}') AS base_date")
    eom_date = spark.sql("SELECT DATE_ADD(DATE_TRUNC('month', base_date), -1) FROM tmp_run_date").collect()[0][0]

# --- Format strings ---
run_date_str = eom_date.strftime('%Y%m%d')
run_date = eom_date.strftime('%Y-%m-%d')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

# --- Debug print ---
print(f"Report {report_name} is running with: portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}, now = {now_datetime_str}")

# --- Optional Spark setup for future date refs ---
spark.sql(f"CREATE OR REPLACE TEMP VIEW tmp_date AS SELECT DATE('{eom_date}') AS base_date")


print(f"Run_date is", run_date)
print(f"eom_date is", eom_date)

## Report Generation

In [0]:
# Define the end of previous month based on the report generation date
eomDate = eom_date
#Below you can find the previous version of the reference month
#RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{eomDate}'), CAST('{eomDate}' AS DATE)) AS DATE)").collect()[0][0]
#Below the new version
RefMonth = spark.sql(f"SELECT date_format(TRUNC(DATE('{eomDate}'), 'MM'), 'yyyy-MM-dd') AS first_day").collect()[0][0]

# Define the first and last dates of the selected reporting month
FirstDate = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

LastDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}'
""").collect()[0][0]

# Define the last business day of the selected reporting month
LastWorkingDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

# Define the second business day of the selected reporting month
SecondWorking = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{RefMonth}' AND BusinessDayOfMonth = 2
""").collect()[0][0]

# Define the first and second business days of the following month
# FirstWorkingNextMonth is the date that acts as the snapshot date for the report
# using as reference the on-premise logic of EffectiveDate <= @FirstWorkingNextMonth AND @FirstWorkingNextMonth < EndDate
FirstWorkingNextMonth = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth}'), CAST('{RefMonth}' AS DATE))) AS DATE)
    AND BusinessDayOfMonth = 1
""").collect()[0][0]

SecondWorkingNextMonth = spark.sql(f"""
    SELECT MIN(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth}'), CAST('{RefMonth}' AS DATE))) AS DATE)
    AND BusinessDayOfMonth = 2
""").collect()[0][0]

#Debugging 
'''
print("RefMonth:", RefMonth)
print("EOMDate:", eomDate)
print("First Date:", FirstDate)
print("Last Date:", LastDate)
print("Last Working Date:", LastWorkingDate)
print("Second Working Date:", SecondWorking)
print("First Working Date Next Month:", FirstWorkingNextMonth)
print("Second Working Date Next Month:", SecondWorkingNextMonth)
'''


In [0]:
# Create the dexposure temporary view
dexposure = spark.sql(f"""
                      select *
                      from `{env_prefix}-cdh`.derivative.dexposure
                      where BusinessDate = {FirstWorkingNextMonth!r} and sourcesystem='profits'
                      and isactiveflag = true and PortfolioName ilike {repr(portfolioName)}
                      union
                      select *
                      from `{env_prefix}-cdh`.derivative.dexposure
                      where BusinessDate <= {FirstWorkingNextMonth!r} and sourcesystem='profits'
                      and isactiveflag = false and PortfolioName ilike {repr(portfolioName)}
                      """)
 
dexposure.createOrReplaceTempView("dexposure")
# Create the dparty temporary view
dparty = spark.sql(f"""
                     select DISTINCT CDI, CASE WHEN L3869Flag = true THEN 'Y' ELSE 'N' END AS L3869Flag, TaxRegistrationNumber AS TaxNumber
                     from `{env_prefix}-cdh`.derivative.dparty
                     where BusinessDate = {FirstWorkingNextMonth!r} and sourcesystem='profits'
                     and PortfolioName ilike {repr(portfolioName)}
                     """)
 
dparty.createOrReplaceTempView("dparty")
 
# Create the Hoist_Secured_Perimeter temporary view
Hoist_Secured_Perimeter = spark.sql(f"""
                      select *
                      from `{env_prefix}-ReportInputFiles`.internal.secured_perimeter
                      where ReferenceDate = (SELECT MAX(ReferenceDate) FROM `{env_prefix}-ReportInputFiles`.internal.secured_perimeter WHERE ReferenceDate <= {FirstWorkingNextMonth!r} and PortfolioName ilike {repr(portfolioName)})
                      and PortfolioName ilike {repr(portfolioName)}
                      """)
 
Hoist_Secured_Perimeter.createOrReplaceTempView("Hoist_Secured_Perimeter")

# Create the RestructuredCustomerSegment temporary view
RestructuredCustomerSegment = spark.sql(f"""
                                        SELECT 
                                            AccountNumber 
                                           ,Segment
                                        FROM `{env_prefix}-reportinputfiles`.internal.cell_sustomersegment_perimeter
                                        WHERE Segment = 'Tranche B' OR Segment = 'Tranche A'
                                        UNION
                                        SELECT  
                                            r.NewExposureKey
                                            ,c.Segment 
                                        FROM `{env_prefix}-reportinputfiles`.internal.cell_sustomersegment_perimeter c
                                        INNER JOIN `p-cdh`.primary.restructuring r
                                        ON c.AccountNumber = r.OldExposureKey
                                        """)

RestructuredCustomerSegment.createOrReplaceTempView("RestructuredCustomerSegment")

# Create the Segment temporary view
# Segment = spark.sql(f"""
#                       SELECT
#                           CDI
#                          ,Segment
#                         FROM `p-cdh`.`primary`.`party`
#                         WHERE 1=1
#                         AND SourceSystem='qc'
#                         AND BusinessDate> '{RefMonth}'
#                          """)

# Segment.createOrReplaceTempView("Segment")

# Create the Segment temporary view
Segment = spark.sql(f"""
    SELECT
        CDI,
        Segment
    FROM `p-cdh`.`primary`.`party`
    WHERE 1=1
      AND BusinessDate > DATE('{RefMonth}')
      AND (
            (
                DATE('{RefMonth}') <= DATE('2026-06-05')
                AND BusinessDate <= DATE('2026-06-05')
                AND SourceSystem = 'qc'
            )
            OR
            (
                BusinessDate > DATE('2026-06-05')
                AND SourceSystem = 'qca'
            )
          )
""")

Segment.createOrReplaceTempView("Segment")

#debug
'''
print(f"Dexposure has: {dexposure.count()} records")
print(f"Dparty has: {dparty.count()} records")
print(f"Secured_Perimeter has: {Hoist_Secured_Perimeter.count()} records")
'''

## Collections of the reporting month

In [0]:
## Sheet 1 for the collections of the reporting month

if portfolioName != 'Cell':

    Collections_mmyy_temp = spark.sql(f"""
        SELECT DISTINCT
              a.PortfolioName AS Portfolio
            , a.ExposureKey AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag AS Law3869
            , t.DCollectionKey AS CollectionId
            , MONTH(t.ActualDate) AS ReferenceMonth
            , YEAR(t.ActualDate) AS ReferenceYear
            , '{SecondWorkingNextMonth}' AS ReportingDate
            , t.PaymentTypeGroup
            , CASE 
                WHEN t.PaymentTypeEng IN (
                    'DISPOSAL COLLECTIONS - AUCTIONS',
                    'PRE-CLOSING CASH IN COURT',
                    'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                    'GROSS / FORECLOSURE - AUCTION SALE',
                    'GROSS /FORECLOSURE-REPOSESSION',
                    'PG COLLECT./FORECLOSURE-AUCTION SALE',
                    'PG COLLECT./FORECLOSURE-REPOSESSION',
                    'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                    'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                    'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                    'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                    'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                    'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                    'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                    'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                    'AUCTION PROCEEDS_UNSECURED CLAIMS_REO'
                ) THEN 'Auction'
                ELSE 'Cash'
              END AS grouped_payment_type
            , t.PaymentTypeEng AS PaymentType
            , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
            , a.AgreementKey AS ContractId
        FROM `{env_prefix}-cdh`.derivative.dcollection t
        INNER JOIN dexposure a
            ON a.ExposureKey = t.ExposureKey
           AND a.PortfolioName = t.PortfolioName
        LEFT JOIN dparty cu
            ON cu.CDI = a.CDI
        LEFT JOIN Hoist_Secured_Perimeter secured
            ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
           AND a.PortfolioName = secured.PortfolioName
        WHERE t.PortfolioName ILIKE {repr(portfolioName)}
          AND t.BusinessDate = '{FirstWorkingNextMonth}'
          AND FinalReason = 'Matched'
          AND COALESCE(t.MatchedDate, DATE('1900-01-01')) <= '{SecondWorkingNextMonth}'
          AND t.ActualDate BETWEEN '{FirstDate}' AND '{LastDate}'
    """)

    Collections_mmyy_temp.createOrReplaceTempView("Collections_mmyy_temp")

    secured_contracts = spark.sql(f"""
        SELECT DISTINCT 
            res.OldAgreementKey AS ContractId,
            c.AccountNumber
        FROM Collections_mmyy_temp c
        LEFT JOIN `{env_prefix}-cdh`.primaryactive.restructuring res
            ON c.ContractId = res.oldAgreementKey
        LEFT JOIN `{env_prefix}-reportinputfiles`.internal.secured_perimeter secured
            ON regexp_replace(res.OLDExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
        WHERE res.NewExposureKey = c.AccountNumber
          AND secured.LoanId IS NOT NULL

        UNION 

        SELECT DISTINCT 
            ContractId,
            AccountNumber
        FROM Collections_mmyy_temp
        WHERE Secured_Flag = 'Y'
    """)

    secured_contracts.createOrReplaceTempView("secured_contracts")

    Collections_mmyy = spark.sql(f"""
        SELECT DISTINCT
              s1.Portfolio
            , s1.AccountNumber
            , s1.RecordDate
            , s1.ActualDate
            , s1.PaymentAmount
            , s1.CDI
            , s1.TaxNumber
            , s1.AssetSegment
            , s1.Law3869
            , s1.CollectionId
            , s1.ReferenceMonth
            , s1.ReferenceYear
            , s1.ReportingDate
            , s1.PaymentTypeGroup
            , s1.grouped_payment_type
            , s1.PaymentType
            , CASE WHEN s2.ContractId IS NOT NULL THEN 'Y' ELSE 'N' END AS SecuredFlag
            , s1.ContractId
        FROM Collections_mmyy_temp s1
        LEFT JOIN secured_contracts s2
          ON s1.ContractId = s2.ContractId
         AND s1.AccountNumber = s2.AccountNumber
    """)

    Collections_mmyy.createOrReplaceTempView("Collections_mmyy")

else:

    Collections_mmyy_temp = spark.sql(f"""
        SELECT DISTINCT
              a.PortfolioName AS Portfolio
            , a.ExposureKey AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag AS Law3869
            , t.DCollectionKey AS CollectionId
            , MONTH(t.ActualDate) AS ReferenceMonth
            , YEAR(t.ActualDate) AS ReferenceYear
            , '{SecondWorkingNextMonth}' AS ReportingDate
            , t.PaymentTypeGroup
            , CASE 
                WHEN t.PaymentTypeEng IN (
                    'DISPOSAL COLLECTIONS - AUCTIONS',
                    'PRE-CLOSING CASH IN COURT',
                    'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                    'GROSS / FORECLOSURE - AUCTION SALE',
                    'GROSS /FORECLOSURE-REPOSESSION',
                    'PG COLLECT./FORECLOSURE-AUCTION SALE',
                    'PG COLLECT./FORECLOSURE-REPOSESSION',
                    'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                    'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                    'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                    'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                    'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                    'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                    'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                    'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                    'AUCTION PROCEEDS_UNSECURED CLAIMS_REO'
                ) THEN 'Auction'
                ELSE 'Cash'
              END AS grouped_payment_type
            , t.PaymentTypeEng AS PaymentType
            , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
            , a.AgreementKey AS ContractId
            , CASE 
                WHEN custseg.Segment IS NOT NULL THEN custseg.Segment
                WHEN seg.Segment IS NOT NULL AND (seg.Segment LIKE '%-B%' OR seg.Segment LIKE '%-Β%')
                     THEN 'Tranche B'
                ELSE 'Tranche A'
              END AS TrancheSegment
        FROM `{env_prefix}-cdh`.derivative.dcollection t
        INNER JOIN dexposure a
            ON a.ExposureKey = t.ExposureKey
           AND a.PortfolioName = t.PortfolioName
        LEFT JOIN dparty cu
            ON cu.CDI = a.CDI
        LEFT JOIN RestructuredCustomerSegment custseg
            ON custseg.AccountNumber = a.ExposureKey
        LEFT JOIN Segment seg
            ON seg.CDI = t.CDI
        LEFT JOIN Hoist_Secured_Perimeter secured
            ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
           AND a.PortfolioName = secured.PortfolioName
        WHERE t.PortfolioName ILIKE {repr(portfolioName)}
          AND t.BusinessDate = '{FirstWorkingNextMonth}'
          AND FinalReason = 'Matched'
          AND COALESCE(CAST(t.MatchedDate AS DATE), DATE('1900-01-01')) <= '{SecondWorkingNextMonth}'
          AND t.ActualDate BETWEEN '{FirstDate}' AND '{LastDate}'
    """)

    Collections_mmyy_temp.createOrReplaceTempView("Collections_mmyy_temp")

    secured_contracts = spark.sql("""
        SELECT DISTINCT ContractId 
        FROM Collections_mmyy_temp 
        WHERE Secured_Flag = 'Y'
    """)

    secured_contracts.createOrReplaceTempView("secured_contracts")

    Collections_mmyy = spark.sql("""
        SELECT DISTINCT
              s1.Portfolio
            , s1.AccountNumber
            , s1.RecordDate
            , s1.ActualDate
            , s1.PaymentAmount
            , s1.CDI
            , s1.TaxNumber
            , s1.AssetSegment
            , s1.Law3869
            , s1.CollectionId
            , s1.ReferenceMonth
            , s1.ReferenceYear
            , s1.ReportingDate
            , s1.PaymentTypeGroup
            , s1.grouped_payment_type
            , s1.PaymentType
            , CASE 
                WHEN s1.ContractId = s2.ContractId THEN 'Y'
                WHEN s1.TrancheSegment = 'Tranche A' THEN 'N-Segment A'
                WHEN s1.TrancheSegment = 'Tranche B' THEN 'N-Segment B'
              END AS SecuredFlag
            , s1.ContractId
        FROM Collections_mmyy_temp s1
        LEFT JOIN secured_contracts s2
          ON s1.ContractId = s2.ContractId
    """)

    Collections_mmyy.createOrReplaceTempView("Collections_mmyy")

# Debug counts
#print(f"Collections_mmyy_temp: {spark.table('Collections_mmyy_temp').count()}")
#print(f"secured_contracts: {spark.table('secured_contracts').count()}")
#print(f"Collections_mmyy: {spark.table('Collections_mmyy').count()}")

In [0]:
## Sheet 1 for the collections of the reporting month

if portfolioName != 'Cell':

    Collections_mmyy_temp = spark.sql(f"""SELECT DISTINCT
              a.PortfolioName AS Portfolio
            , a.ExposureKey AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount                           AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag                       AS Law3869
            , t.DCollectionKey                    AS CollectionId
            , MONTH(t.ActualDate)                AS ReferenceMonth
            , YEAR(t.ActualDate)                 AS ReferenceYear
            , '{SecondWorkingNextMonth}'         AS ReportingDate 
            , t.PaymentTypeGroup
            , CASE WHEN t.PaymentTypeEng IN (
                        'DISPOSAL COLLECTIONS - AUCTIONS',
                        'PRE-CLOSING CASH IN COURT',
                        'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                        'GROSS / FORECLOSURE - AUCTION SALE',
                        'GROSS /FORECLOSURE-REPOSESSION',
                        'PG COLLECT./FORECLOSURE-AUCTION SALE',
                        'PG COLLECT./FORECLOSURE-REPOSESSION',
                        'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                        'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                        'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                        'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                        'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                        'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                        'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                        'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                        'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                   THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
            , t.PaymentTypeEng AS PaymentType
            , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
            , a.AgreementKey AS ContractId
        FROM `{env_prefix}-cdh`.derivative.dcollection t
        INNER JOIN dexposure a
          ON a.ExposureKey = t.ExposureKey
         AND a.PortfolioName = t.PortfolioName
        LEFT JOIN dparty cu
          ON cu.CDI = a.CDI
        LEFT JOIN Hoist_Secured_Perimeter secured
          ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
         AND a.PortfolioName = secured.PortfolioName
        WHERE  t.PortfolioName ILIKE {repr(portfolioName)}
          AND t.BusinessDate = '{FirstWorkingNextMonth}'
          AND FinalReason = 'Matched'
          AND COALESCE(t.MatchedDate, DATE('1900-01-01')) <= '{SecondWorkingNextMonth}'
          AND t.ActualDate BETWEEN '{FirstDate}' AND '{LastDate}'
    """)

    Collections_mmyy_temp.createOrReplaceTempView("Collections_mmyy_temp")

    # --- Fixed secured_contracts logic (ContractId + AccountNumber) ---
    secured_contracts = spark.sql(f"""
        SELECT DISTINCT 
            res.OldAgreementKey AS ContractId,
            c.AccountNumber
        FROM Collections_mmyy_temp c
        LEFT JOIN `{env_prefix}-cdh`.primaryactive.restructuring res
          ON c.ContractId = res.oldAgreementKey
        LEFT JOIN `{env_prefix}-reportinputfiles`.internal.secured_perimeter secured
          ON regexp_replace(res.OLDExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
        WHERE res.NewExposureKey = c.AccountNumber
          AND secured.LoanId IS NOT NULL

        UNION 

        SELECT DISTINCT 
            ContractId,
            AccountNumber
        FROM Collections_mmyy_temp
        WHERE Secured_Flag = 'Y'
    """)

    secured_contracts.createOrReplaceTempView("secured_contracts")

    Collections_mmyy = spark.sql(f"""
        SELECT DISTINCT
              s1.Portfolio
            , s1.AccountNumber
            , s1.RecordDate
            , s1.ActualDate
            , s1.PaymentAmount
            , s1.CDI
            , s1.TaxNumber
            , s1.AssetSegment
            , s1.Law3869
            , s1.CollectionId
            , s1.ReferenceMonth
            , s1.ReferenceYear
            , s1.ReportingDate
            , s1.PaymentTypeGroup
            , s1.grouped_payment_type
            , s1.PaymentType
            , CASE WHEN s2.ContractId IS NOT NULL THEN 'Y' ELSE 'N' END AS SecuredFlag
            , s1.ContractId
        FROM Collections_mmyy_temp s1
        LEFT JOIN secured_contracts s2
          ON s1.ContractId = s2.ContractId
         AND s1.AccountNumber = s2.AccountNumber
    """)

    Collections_mmyy.createOrReplaceTempView("Collections_mmyy")

else:

    Collections_mmyy_temp = spark.sql(f"""SELECT DISTINCT
              a.PortfolioName AS Portfolio
            , a.ExposureKey AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag AS Law3869
            , t.DCollectionKey AS CollectionId
            , MONTH(t.ActualDate) AS ReferenceMonth
            , YEAR(t.ActualDate) AS ReferenceYear
            , '{SecondWorkingNextMonth}' AS ReportingDate
            , t.PaymentTypeGroup
            , CASE WHEN t.PaymentTypeEng IN (
                        'DISPOSAL COLLECTIONS - AUCTIONS',
                        'PRE-CLOSING CASH IN COURT',
                        'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                        'GROSS / FORECLOSURE - AUCTION SALE',
                        'GROSS /FORECLOSURE-REPOSESSION',
                        'PG COLLECT./FORECLOSURE-AUCTION SALE',
                        'PG COLLECT./FORECLOSURE-REPOSESSION',
                        'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                        'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                        'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                        'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                        'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                        'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                        'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                        'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                        'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                   THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
            , t.PaymentTypeEng AS PaymentType
            , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
            , a.AgreementKey AS ContractId
            , CASE 
                 WHEN custseg.Segment IS NOT NULL THEN custseg.Segment
                 WHEN seg.Segment IS NOT NULL AND (seg.Segment LIKE '%-B%' OR seg.Segment LIKE '%-Β%') THEN 'Tranche B'
                 ELSE 'Tranche A'
              END AS TrancheSegment
        FROM `{env_prefix}-cdh`.derivative.dcollection t
        INNER JOIN dexposure a
          ON a.ExposureKey = t.ExposureKey
         AND a.PortfolioName = t.PortfolioName
        LEFT JOIN dparty cu ON cu.CDI = a.CDI
        LEFT JOIN RestructuredCustomerSegment custseg ON custseg.AccountNumber = a.ExposureKey
        LEFT JOIN Segment seg ON seg.CDI = t.CDI
        LEFT JOIN Hoist_Secured_Perimeter secured
          ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
         AND a.PortfolioName = secured.PortfolioName
        WHERE  t.PortfolioName ILIKE {repr(portfolioName)}
          AND t.BusinessDate = '{FirstWorkingNextMonth}'
          AND FinalReason = 'Matched'
          AND COALESCE(t.MatchedDate, DATE('1900-01-01')) <= '{SecondWorkingNextMonth}'
          AND t.ActualDate BETWEEN '{FirstDate}' AND '{LastDate}'
    """)

    Collections_mmyy_temp.createOrReplaceTempView("Collections_mmyy_temp")

    secured_contracts = spark.sql(f"""
        SELECT DISTINCT ContractId, AccountNumber
        FROM Collections_mmyy_temp
        WHERE Secured_Flag = 'Y'
    """)

    secured_contracts.createOrReplaceTempView("secured_contracts")

    Collections_mmyy = spark.sql(f"""
        SELECT DISTINCT
              s1.Portfolio
            , s1.AccountNumber
            , s1.RecordDate
            , s1.ActualDate
            , s1.PaymentAmount
            , s1.CDI
            , s1.TaxNumber
            , s1.AssetSegment
            , s1.Law3869
            , s1.CollectionId
            , s1.ReferenceMonth
            , s1.ReferenceYear
            , s1.ReportingDate
            , s1.PaymentTypeGroup
            , s1.grouped_payment_type
            , s1.PaymentType
            , CASE 
                 WHEN s2.ContractId IS NOT NULL THEN 'Y'
                 WHEN s1.TrancheSegment = 'Tranche A' THEN 'N-Segment A'
                 WHEN s1.TrancheSegment = 'Tranche B' THEN 'N-Segment B'
                 ELSE 'N'
              END AS SecuredFlag
            , s1.ContractId
        FROM Collections_mmyy_temp s1
        LEFT JOIN secured_contracts s2
          ON s1.ContractId = s2.ContractId
         AND s1.AccountNumber = s2.AccountNumber
    """)

    Collections_mmyy.createOrReplaceTempView("Collections_mmyy")


## Collections of previous months

In [0]:
## Sheet 2 for the collections of previous months

if portfolioName != 'Cell':

  Collections_mmyy_prevmonths_temp = spark.sql(f"""SELECT DISTINCT
              a.PortfolioName            AS Portfolio
            , a.ExposureKey              AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount                   AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag               AS Law3869
            , t.DCollectionKey            AS CollectionId	                                                   
            , MONTH(t.ActualDate)        AS ReferenceMonth
            , YEAR(t.ActualDate)         AS ReferenceYear
            , '{SecondWorkingNextMonth}' AS ReportingDate 
            , t.PaymentTypeGroup
            ,CASE WHEN t.PaymentTypeEng IN ('DISPOSAL COLLECTIONS - AUCTIONS',
                                            'PRE-CLOSING CASH IN COURT',
                                            'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                                            'GROSS / FORECLOSURE - AUCTION SALE',
                                            'GROSS /FORECLOSURE-REPOSESSION',
                                            'PG COLLECT./FORECLOSURE-AUCTION SALE',
                                            'PG COLLECT./FORECLOSURE-REPOSESSION',
                                            'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                                            'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                                            'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                                            'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                                            'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                                            'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                                            'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                                            'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                                            'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                                          THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
              , t.PaymentTypeEng AS PaymentType
              , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
              , t.MatchedDate
              , a.AgreementKey AS ContractId
              FROM `{env_prefix}-cdh`.derivative.dcollection t
              INNER JOIN dexposure   a
              ON a.ExposureKey = t.ExposureKey
              AND a.PortfolioName = t.PortfolioName
              LEFT JOIN dparty  cu
              ON cu.CDI = a.CDI
              LEFT JOIN Hoist_Secured_Perimeter secured
              ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
              AND a.PortfolioName = secured.PortfolioName
              WHERE t.PortfolioName ilike {repr(portfolioName)}
              AND t.BusinessDate = '{FirstWorkingNextMonth}'
              AND FinalReason = 'Matched'
              /*commented due to DBTC-1413*/
              /*AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END > '{SecondWorking}'
              AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END <= '{SecondWorkingNextMonth}'*/
              AND t.ActualDate < '{FirstDate}'
              AND 
                  ( 
                    (CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END > '{SecondWorking}'
                    AND 
                    CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth}'
                    )
                  OR 
                    (t.MatchedDate >= '{FirstDate}' 
                    AND t.MatchedDate <= '{LastDate}'
                    AND t.PortfolioName = 'Cell' --added later for Cell
                    ) 
                  )
            """)


  Collections_mmyy_prevmonths_temp.createOrReplaceTempView("Collections_mmyy_prevmonths_temp")

  secured_contracts_mmyy_prevmonths = spark.sql(f"""
                        SELECT DISTINCT ContractId FROM Collections_mmyy_prevmonths_temp WHERE Secured_Flag = 'Y'
                        """)

  secured_contracts_mmyy_prevmonths.createOrReplaceTempView("secured_contracts_mmyy_prevmonths")

  Collections_mmyy_prevmonths = spark.sql(f"""
                        SELECT DISTINCT
                              Portfolio
                              ,AccountNumber
                              ,RecordDate
                              ,ActualDate
                              ,PaymentAmount
                              ,CDI
                              ,TaxNumber
                              ,AssetSegment
                              ,Law3869
                              ,CollectionId
                              ,ReferenceMonth
                              ,ReferenceYear
                              ,ReportingDate
                              ,PaymentTypeGroup
                              ,grouped_payment_type
                              ,PaymentType
                              ,CASE WHEN S1.ContractId=S2.ContractId THEN 'Y' ELSE 'N' END AS SecuredFlag
                              ,MatchedDate
                              ,s1.ContractId
                          FROM Collections_mmyy_prevmonths_temp s1
                          LEFT JOIN secured_contracts_mmyy_prevmonths s2
                          ON s1.ContractId = s2.ContractId
                        """)

  Collections_mmyy_prevmonths.createOrReplaceTempView("Collections_mmyy_prevmonths")

else:

  Collections_mmyy_prevmonths_temp = spark.sql(f"""SELECT DISTINCT
              a.PortfolioName            AS Portfolio
            , a.ExposureKey              AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount                   AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag               AS Law3869
            , t.DCollectionKey            AS CollectionId	                                                   
            , MONTH(t.ActualDate)        AS ReferenceMonth
            , YEAR(t.ActualDate)         AS ReferenceYear
            , '{SecondWorkingNextMonth}' AS ReportingDate 
            , t.PaymentTypeGroup
            ,CASE WHEN t.PaymentTypeEng IN ('DISPOSAL COLLECTIONS - AUCTIONS',
                                            'PRE-CLOSING CASH IN COURT',
                                            'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                                            'GROSS / FORECLOSURE - AUCTION SALE',
                                            'GROSS /FORECLOSURE-REPOSESSION',
                                            'PG COLLECT./FORECLOSURE-AUCTION SALE',
                                            'PG COLLECT./FORECLOSURE-REPOSESSION',
                                            'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                                            'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                                            'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                                            'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                                            'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                                            'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                                            'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                                            'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                                            'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                                          THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
              , t.PaymentTypeEng AS PaymentType
              , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
              , t.MatchedDate
              , a.AgreementKey AS ContractId
              --6/8/25: added for Cell
              , CASE WHEN custseg.Segment IS NOT NULL THEN custseg.Segment
                    WHEN seg.Segment IS NOT NULL AND (seg.Segment like '%-B%' or seg.Segment like '%-Β%')  THEN 'Tranche B'
                    ELSE 'Tranche A'											
                END AS TrancheSegment
              FROM `{env_prefix}-cdh`.derivative.dcollection t
              INNER JOIN dexposure   a
              ON a.ExposureKey = t.ExposureKey
              AND a.PortfolioName = t.PortfolioName
              LEFT JOIN dparty  cu
              ON cu.CDI = a.CDI
              --6/8/25: added for Cell
              LEFT JOIN RestructuredCustomerSegment custseg
              ON custseg.AccountNumber = a.ExposureKey
              LEFT JOIN Segment seg
              ON seg.CDI=t.CDI
              LEFT JOIN Hoist_Secured_Perimeter secured
              ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
              AND a.PortfolioName = secured.PortfolioName
              WHERE t.PortfolioName ilike {repr(portfolioName)}
              AND t.BusinessDate = '{FirstWorkingNextMonth}'
              AND FinalReason = 'Matched'
              /*commented due to DBTC-1413*/
              /*AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END > '{SecondWorking}'
              AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END <= '{SecondWorkingNextMonth}'*/
              AND t.ActualDate < '{FirstDate}'
              AND 
                  ( 
                    (CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END > '{SecondWorking}'
                    AND 
                    CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth}'
                    )
                  OR 
                    (t.MatchedDate >= '{FirstDate}' 
                    AND t.MatchedDate <= '{LastDate}'
                    AND t.PortfolioName = 'Cell' --added later for Cell
                    ) 
                  )
            """)


  Collections_mmyy_prevmonths_temp.createOrReplaceTempView("Collections_mmyy_prevmonths_temp")

  secured_contracts_mmyy_prevmonths = spark.sql(f"""
                        SELECT DISTINCT ContractId FROM Collections_mmyy_prevmonths_temp WHERE Secured_Flag = 'Y'
                        """)

  secured_contracts_mmyy_prevmonths.createOrReplaceTempView("secured_contracts_mmyy_prevmonths")

  Collections_mmyy_prevmonths = spark.sql(f"""
                        SELECT DISTINCT
                              Portfolio
                              ,AccountNumber
                              ,RecordDate
                              ,ActualDate
                              ,PaymentAmount
                              ,CDI
                              ,TaxNumber
                              ,AssetSegment
                              ,Law3869
                              ,CollectionId
                              ,ReferenceMonth
                              ,ReferenceYear
                              ,ReportingDate
                              ,PaymentTypeGroup
                              ,grouped_payment_type
                              ,PaymentType
                              ,CASE WHEN S1.ContractId=S2.ContractId THEN 'Y'
                                    WHEN s1.TrancheSegment='Tranche A' THEN 'N-Segment A'
                                    WHEN s1.TrancheSegment='Tranche B' THEN 'N-Segment B'
                              END AS SecuredFlag
                              ,MatchedDate
                              ,s1.ContractId
                          FROM Collections_mmyy_prevmonths_temp s1
                          LEFT JOIN secured_contracts_mmyy_prevmonths s2
                          ON s1.ContractId = s2.ContractId
                        """)

  Collections_mmyy_prevmonths.createOrReplaceTempView("Collections_mmyy_prevmonths")


#Debug for sheet 2
'''
print(f"Collections_mmyy_prevmonths_temp has: {spark.table('Collections_mmyy_prevmonths_temp').count()} rows")
print(f"secured_contracts_mmyy_prevmonths has: {spark.table('secured_contracts_mmyy_prevmonths').count()} rows")
print(f"Collections_mmyy_prevmonths has: {spark.table('Collections_mmyy_prevmonths').count()} rows")
'''

## Collections of the subsequent month

In [0]:
## Sheet 3 for the collections of the subsequent month

if portfolioName != 'Cell':

    RefMonth_mmyynext =  spark.sql(f"""SELECT ADD_MONTHS(DATE_TRUNC('month', CAST('{run_date}' AS DATE)), 1)""").collect()[0][0]
    
    FirstDate_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}'
    """).collect()[0][0]
    
    LastDate_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}'
    """).collect()[0][0]
    
    LastWorkingDate_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}' AND IsBusinessDay = 1
    """).collect()[0][0]
    
    SecondWorking_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}' AND BusinessDayOfMonth = 2
    """).collect()[0][0]
    
    FirstWorkingNextMonth_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth_mmyynext}'), CAST('{RefMonth_mmyynext}' AS DATE))) AS DATE)
        AND BusinessDayOfMonth = 1
    """).collect()[0][0]
    
    SecondWorkingNextMonth_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth_mmyynext}'), CAST('{RefMonth_mmyynext}' AS DATE))) AS DATE)
        AND BusinessDayOfMonth = 2
    """).collect()[0][0]
    
    # Create the dexposure temporary view
    dexposure_mmyynext = spark.sql(f"""
                        select *
                        from `{env_prefix}-cdh`.derivative.dexposure
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate = {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and isactiveflag = true and PortfolioName ilike {repr(portfolioName)}
                        union
                        select *
                        from `{env_prefix}-cdh`.derivative.dexposure
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate <= {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and isactiveflag = false and PortfolioName ilike {repr(portfolioName)}
                        """)
    
    dexposure_mmyynext.createOrReplaceTempView("dexposure_mmyynext")
    
    # Create the dparty temporary view
    dparty_mmyynext = spark.sql(f"""
                        select DISTINCT CDI, CASE WHEN L3869Flag = true THEN 'Y' ELSE 'N' END AS L3869Flag, TaxRegistrationNumber AS TaxNumber
                        from `{env_prefix}-cdh`.derivative.dparty
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate = {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and PortfolioName ilike {repr(portfolioName)}
                        """)
    
    dparty_mmyynext.createOrReplaceTempView("dparty_mmyynext")
    
    Collections_mmyynext_prevmonths_temp = spark.sql(f"""SELECT DISTINCT
                a.PortfolioName                        AS Portfolio
            , a.ExposureKey                          AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount                               AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag                           AS Law3869
            , t.DCollectionKey                        AS CollectionId                                                    
            , MONTH(t.ActualDate)                    AS ReferenceMonth
            , YEAR(t.ActualDate)                     AS ReferenceYear
            , '{SecondWorkingNextMonth_mmyynext}'    AS ReportingDate
            , t.PaymentTypeGroup
            ,CASE WHEN t.PaymentTypeEng IN ('DISPOSAL COLLECTIONS - AUCTIONS',
                                                'PRE-CLOSING CASH IN COURT',
                                                'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                                                'GROSS / FORECLOSURE - AUCTION SALE',
                                                'GROSS /FORECLOSURE-REPOSESSION',
                                                'PG COLLECT./FORECLOSURE-AUCTION SALE',
                                                'PG COLLECT./FORECLOSURE-REPOSESSION',
                                                'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                                                'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                                                'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                                                'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                                                'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                                                'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                                                'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                                                'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                                                'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                                            THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
                , t.PaymentTypeEng AS PaymentType
                , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
                , t.MatchedDate
                , a.AgreementKey AS ContractId
                FROM `{env_prefix}-cdh`.derivative.dcollection  t
                INNER JOIN dexposure_mmyynext  a
                ON a.ExposureKey = t.ExposureKey
                AND a.PortfolioName = t.PortfolioName
                LEFT JOIN dparty_mmyynext cu
                ON cu.CDI = a.CDI
                LEFT JOIN Hoist_Secured_Perimeter secured
                ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
                AND a.PortfolioName = secured.PortfolioName
                WHERE t.PortfolioName ilike {repr(portfolioName)}
                --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                AND t.BusinessDate = '{SecondWorking_mmyynext}' --- the code says <= but the initial code wrote =
                AND FinalReason = 'Matched'
                /*commented due to DBTC-1413*/
                /*AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END > '{SecondWorking_mmyynext}'
                AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END <= '{SecondWorkingNextMonth_mmyynext}'*/
                AND t.ActualDate < '{FirstDate_mmyynext}'
                AND
                    (
                        (CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END > '{SecondWorking_mmyynext}'
                        AND
                        CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth_mmyynext}')
                    OR
                        (t.MatchedDate >= '{FirstDate_mmyynext}'
                        AND
                        t.MatchedDate <= '{LastDate_mmyynext}'
                        AND t.PortfolioName = 'Cell')
                    )
                """)
    
    Collections_mmyynext_prevmonths_temp.createOrReplaceTempView("Collections_mmyynext_prevmonths_temp")
    
    secured_contracts_mmyynext_prevmonths = spark.sql(f"""
                        SELECT DISTINCT ContractId FROM Collections_mmyynext_prevmonths_temp WHERE Secured_Flag = 'Y'
                        """)
    
    secured_contracts_mmyynext_prevmonths.createOrReplaceTempView("secured_contracts_mmyynext_prevmonths")
    
    Collections_mmyynext_prevmonths = spark.sql(f"""
                        SELECT DISTINCT
                                 Portfolio
                                ,AccountNumber
                                ,RecordDate
                                ,ActualDate
                                ,PaymentAmount
                                ,CDI
                                ,TaxNumber
                                ,AssetSegment
                                ,Law3869
                                ,CollectionId
                                ,ReferenceMonth
                                ,ReferenceYear
                                ,ReportingDate
                                ,PaymentTypeGroup
                                ,grouped_payment_type
                                ,PaymentType
                                ,CASE WHEN S1.ContractId=S2.ContractId THEN 'Y' ELSE 'N' END AS SecuredFlag
                                ,MatchedDate
                                ,s1.ContractId
                            FROM Collections_mmyynext_prevmonths_temp s1
                            LEFT JOIN secured_contracts_mmyynext_prevmonths s2
                            ON s1.ContractId = s2.ContractId
                        """)
else:

    RefMonth_mmyynext =  spark.sql(f"""SELECT ADD_MONTHS(DATE_TRUNC('month', CAST('{run_date}' AS DATE)), 1)""").collect()[0][0]
    
    FirstDate_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}'
    """).collect()[0][0]
    
    LastDate_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}'
    """).collect()[0][0]
    
    LastWorkingDate_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}' AND IsBusinessDay = 1
    """).collect()[0][0]
    
    SecondWorking_mmyynext = spark.sql(f"""
        SELECT MAX(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = '{RefMonth_mmyynext}' AND BusinessDayOfMonth = 2
    """).collect()[0][0]
    
    FirstWorkingNextMonth_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth_mmyynext}'), CAST('{RefMonth_mmyynext}' AS DATE))) AS DATE)
        AND BusinessDayOfMonth = 1
    """).collect()[0][0]
    
    SecondWorkingNextMonth_mmyynext = spark.sql(f"""
        SELECT MIN(CalendarDate)
        FROM `{env_prefix}-cdh`.reference.ref_calendar
        WHERE FirstDateOfMonth = CAST(DATEADD(MONTH, 1, DATEADD(DAY, 1 - DAY('{RefMonth_mmyynext}'), CAST('{RefMonth_mmyynext}' AS DATE))) AS DATE)
        AND BusinessDayOfMonth = 2
    """).collect()[0][0]
    
    # Create the dexposure temporary view
    dexposure_mmyynext = spark.sql(f"""
                        select *
                        from `{env_prefix}-cdh`.derivative.dexposure
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate = {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and isactiveflag = true and PortfolioName ilike {repr(portfolioName)}
                        union
                        select *
                        from `{env_prefix}-cdh`.derivative.dexposure
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate <= {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and isactiveflag = false and PortfolioName ilike {repr(portfolioName)}
                        """)
    
    dexposure_mmyynext.createOrReplaceTempView("dexposure_mmyynext")
    
    # Create the dparty temporary view
    dparty_mmyynext = spark.sql(f"""
                        select DISTINCT CDI, CASE WHEN L3869Flag = true THEN 'Y' ELSE 'N' END AS L3869Flag, TaxRegistrationNumber AS TaxNumber
                        from `{env_prefix}-cdh`.derivative.dparty
                        --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                        where BusinessDate = {SecondWorking_mmyynext!r} and sourcesystem='profits'
                        and PortfolioName ilike {repr(portfolioName)}
                        """)
    
    dparty_mmyynext.createOrReplaceTempView("dparty_mmyynext")
    
    Collections_mmyynext_prevmonths_temp = spark.sql(f"""SELECT DISTINCT
                a.PortfolioName                       AS Portfolio
            , a.ExposureKey                          AS AccountNumber
            , t.RecordDate
            , t.ActualDate
            , t.Amount                               AS PaymentAmount
            , cu.CDI
            , cu.TaxNumber
            , a.AssetSegment
            , cu.L3869Flag                           AS Law3869
            , t.DCollectionKey                        AS CollectionId                                                    
            , MONTH(t.ActualDate)                    AS ReferenceMonth
            , YEAR(t.ActualDate)                     AS ReferenceYear
            , '{SecondWorkingNextMonth_mmyynext}'    AS ReportingDate
            , t.PaymentTypeGroup
            , CASE WHEN t.PaymentTypeEng IN ('DISPOSAL COLLECTIONS - AUCTIONS',
                                                'PRE-CLOSING CASH IN COURT',
                                                'GROSS PROCEEDS FORECLOSURES (AUCTION)',
                                                'GROSS / FORECLOSURE - AUCTION SALE',
                                                'GROSS /FORECLOSURE-REPOSESSION',
                                                'PG COLLECT./FORECLOSURE-AUCTION SALE',
                                                'PG COLLECT./FORECLOSURE-REPOSESSION',
                                                'RE COLLATERAL AUCTION PROCEEDS-AO_3RD',
                                                'RE COLL.AUCTION PROCEEDS-3RD PARTY_3RD',
                                                'NON-RE COLLATERAL AUCTION PROCEEDS-AO',
                                                'NON-RE COLLAT.AUCTION PROCEEDS-3RD PARTY',
                                                'AUCTION PROCEEDS_UNSECURED CLAIMS_3RD',
                                                'COLLECT. VIA VEHICLE LIQUIDAT. PROCESS',
                                                'RE COLLATERAL AUCTION PROCEEDS-AO_REO',
                                                'RE COLL. AUCTION PROCEEDS-3RD PARTY_REO',
                                                'AUCTION PROCEEDS_UNSECURED CLAIMS_REO')
                                            THEN 'Auction' ELSE 'Cash' END AS grouped_payment_type
                , t.PaymentTypeEng AS PaymentType
                , CASE WHEN secured.LoanID IS NOT NULL THEN 'Y' ELSE 'N' END AS Secured_Flag
                , t.MatchedDate
                , a.AgreementKey AS ContractId
                --6/8/25: added for Cell
                , CASE WHEN custseg.Segment IS NOT NULL THEN custseg.Segment
                    WHEN seg.Segment IS NOT NULL AND (seg.Segment like '%-B%' or seg.Segment like '%-Β%')  THEN 'Tranche B'
                    ELSE 'Tranche A'											
                END AS TrancheSegment
                FROM `{env_prefix}-cdh`.derivative.dcollection  t
                INNER JOIN dexposure_mmyynext  a
                ON a.ExposureKey = t.ExposureKey
                AND a.PortfolioName = t.PortfolioName
                LEFT JOIN dparty_mmyynext cu
                ON cu.CDI = a.CDI
                --6/8/25: added for Cell
                LEFT JOIN RestructuredCustomerSegment custseg
                ON custseg.AccountNumber = a.ExposureKey
                LEFT JOIN Segment seg
                ON seg.CDI=t.CDI
                LEFT JOIN Hoist_Secured_Perimeter secured
                ON regexp_replace(a.ExposureKey, '[^0-9]', '') = regexp_replace(secured.LoanID, '[^0-9]', '')
                AND a.PortfolioName = secured.PortfolioName
                WHERE t.PortfolioName ilike {repr(portfolioName)}
                --29/05: Initial code had FirstWorkingNextMonth_mmyynext, changed to SecondWorking_mmyynext in order to bring results  
                AND t.BusinessDate = '{SecondWorking_mmyynext}' --- the code says <= but the initial code wrote =
                AND FinalReason = 'Matched'
                /*commented due to DBTC-1413*/
                /*AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END > '{SecondWorking_mmyynext}'
                AND CASE WHEN CAST(t.MatchedDate AS DATE) IS NULL THEN '1900-01-01' ELSE CAST(t.MatchedDate AS DATE) END <= '{SecondWorkingNextMonth_mmyynext}'*/
                AND t.ActualDate < '{FirstDate_mmyynext}'
                AND
                    (
                        (CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END > '{SecondWorking_mmyynext}'
                        AND
                        CASE WHEN t.MatchedDate IS NULL THEN '1900-01-01' ELSE t.MatchedDate END <= '{SecondWorkingNextMonth_mmyynext}')
                    OR
                        (t.MatchedDate >= '{FirstDate_mmyynext}'
                        AND
                        t.MatchedDate <= '{LastDate_mmyynext}'
                        AND t.PortfolioName = 'Cell')
                    )
                """)
    
    Collections_mmyynext_prevmonths_temp.createOrReplaceTempView("Collections_mmyynext_prevmonths_temp")
    
    secured_contracts_mmyynext_prevmonths = spark.sql(f"""
                        SELECT DISTINCT ContractId FROM Collections_mmyynext_prevmonths_temp WHERE Secured_Flag = 'Y'
                        """)
    
    secured_contracts_mmyynext_prevmonths.createOrReplaceTempView("secured_contracts_mmyynext_prevmonths")
    
    Collections_mmyynext_prevmonths = spark.sql(f"""
                        SELECT DISTINCT
                                Portfolio
                                ,AccountNumber
                                ,RecordDate
                                ,ActualDate
                                ,PaymentAmount
                                ,CDI
                                ,TaxNumber
                                ,AssetSegment
                                ,Law3869
                                ,CollectionId
                                ,ReferenceMonth
                                ,ReferenceYear
                                ,ReportingDate
                                ,PaymentTypeGroup
                                ,grouped_payment_type
                                ,PaymentType
                                ,CASE WHEN S1.ContractId=S2.ContractId THEN 'Y'
                                      WHEN s1.TrancheSegment='Tranche A' THEN 'N-Segment A'
                                      WHEN s1.TrancheSegment='Tranche B' THEN 'N-Segment B'
                                 END AS SecuredFlag
                                ,MatchedDate
                                ,s1.ContractId
                            FROM Collections_mmyynext_prevmonths_temp s1
                            LEFT JOIN secured_contracts_mmyynext_prevmonths s2
                            ON s1.ContractId = s2.ContractId
                        """)
 
#Debug sheet 3
'''
Collections_mmyynext_prevmonths.createOrReplaceTempView("Collections_mmyynext_prevmonths")
print(f"RefMonth_mmyynext: {RefMonth_mmyynext}")
print(f"Reference Month (Next): {RefMonth_mmyynext}")
print(f"First Date of Next Period: {FirstDate_mmyynext}")
print(f"Last Date of Next Period: {LastDate_mmyynext}")
print(f"Last Working Date of Next Period: {LastWorkingDate_mmyynext}")
print(f"Second Working Date of Next Period: {SecondWorking_mmyynext}")
print(f"First Working Date of Following Month: {FirstWorkingNextMonth_mmyynext}")
print(f"Second Working Date of Following Month: {SecondWorkingNextMonth_mmyynext}")
print(f"Collections_mmyynext_prevmonths has",{Collections_mmyynext_prevmonths.count(),"rows"})
print(f"secured_contracts_mmyynext_prevmonths has",{secured_contracts_mmyynext_prevmonths.count(),"rows"})
temp_tables = [
    "dexposure_mmyynext",
    "dparty_mmyynext",
    "Collections_mmyynext_prevmonths_temp",
    "secured_contracts_mmyynext_prevmonths",
    "Collections_mmyynext_prevmonths"
]
 
for table in temp_tables:
    count = spark.table(table).count() if spark.catalog.tableExists(table) else 0
    print(f"{table} count: {count}")
'''
 

## Closing data from the previous quarter

In [0]:
## Sheet 4 for the closing data from the previous quarter
PreviousQuarter = spark.sql(f"""
                            SELECT 
                                CASE 
                                    -- WHEN MONTH('{run_date}') IN (1, 2, 3) THEN 4
                                    -- WHEN MONTH('{run_date}') IN (4, 5, 6) THEN 1
                                    -- WHEN MONTH('{run_date}') IN (7, 8, 9) THEN 2
                                    -- ELSE 3
                                    --NikosAnagn: edited so as to export the correct endofquarter excel sheet, at the end of the quarter
                                    WHEN MONTH('{run_date}') IN (1, 2, 12) THEN 4
                                    WHEN MONTH('{run_date}') IN (3, 4, 5) THEN 1
                                    WHEN MONTH('{run_date}') IN (6, 7, 8) THEN 2
                                    WHEN MONTH('{run_date}') IN (9, 10, 11) THEN 3
                                END
                            """).collect()[0][0]

PreviousYear = spark.sql(f"""
                            SELECT 
                                CASE 
                                    WHEN MONTH('{run_date}') IN (1, 2, 3) THEN YEAR('{run_date}') - 1
                                    ELSE YEAR('{run_date}')
                                END
                            """).collect()[0][0]

QuarterEndDate = spark.sql(f"""SELECT DISTINCT LastDateOfQuarter FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarQuarter = '{PreviousQuarter}' AND CalendarYear = '{PreviousYear}'""").collect()[0][0]

QuarterRefMonth = spark.sql(f"""SELECT CAST(DATEADD(DAY, 1 - DAY('{QuarterEndDate}'), CAST('{QuarterEndDate}' AS DATE)) AS DATE)""").collect()[0][0]

QuarterLastWorkingDate = spark.sql(f"""
    SELECT MAX(CalendarDate)
    FROM `{env_prefix}-cdh`.reference.ref_calendar
    WHERE FirstDateOfMonth = '{QuarterRefMonth}' AND IsBusinessDay = 1
""").collect()[0][0]

# Active accounts for the last working date of the previous quarter
Quarterdexposure_temp1 = spark.sql(f"""
                    SELECT *
                    FROM `{env_prefix}-cdh`.derivative.dexposure
                    WHERE BusinessDate = '{QuarterLastWorkingDate}' AND sourcesystem='profits'
                    AND isactiveflag = true AND PortfolioName ilike {repr(portfolioName)}
                    """)

Quarterdexposure_temp1.createOrReplaceTempView("Quarterdexposure_temp1")

# Inactive accounts for the previous quarter
Quarterdexposure_temp2 = spark.sql(f"""
                    SELECT A.*
                    FROM `{env_prefix}-cdh`.derivative.dexposure A
                    LEFT JOIN Quarterdexposure_temp1 B
                    ON A.ExposureKey = B.ExposureKey
                    WHERE A.BusinessDate <= '{QuarterLastWorkingDate}' AND A.sourcesystem='profits'
                    AND A.isactiveflag = false AND a.PortfolioName ilike {repr(portfolioName)}
                    AND B.ExposureKey IS NULL
                    """)

Quarterdexposure_temp2.createOrReplaceTempView("Quarterdexposure_temp2")

# Create the dexposure temporary view
Quarterdexposure = spark.sql(f"""
                    select *
                    from Quarterdexposure_temp1
                    union
                    select *
                    from Quarterdexposure_temp2
                    """)

Quarterdexposure.createOrReplaceTempView("Quarterdexposure")

# Create the Hoist_Secured_Perimeter temporary view
Quarter_Secured_Perimeter = spark.sql(f"""
                    select *
                    from `{env_prefix}-ReportInputFiles`.internal.secured_perimeter
                    where ReferenceDate = (SELECT MAX(ReferenceDate) FROM `{env_prefix}-ReportInputFiles`.internal.secured_perimeter WHERE ReferenceDate <= '{QuarterLastWorkingDate}' and PortfolioName ilike {repr(portfolioName)})
                    and PortfolioName ilike {repr(portfolioName)}
                    """)

Quarter_Secured_Perimeter.createOrReplaceTempView("Quarter_Secured_Perimeter")

temp1 = spark.sql(f"""
                    SELECT DISTINCT l.AgreementKey AS ContractId
                    FROM Quarterdexposure l
                    INNER JOIN Quarter_Secured_Perimeter osb
                    ON regexp_replace(l.ExposureKey, '[^0-9]', '') = regexp_replace(osb.LoanID, '[^0-9]', '')
                    AND l.PortfolioName = osb.PortfolioName
                    """)

temp1.createOrReplaceTempView("temp1")

SecuredPerimeter_Quarter = spark.sql(f"""
                    SELECT DISTINCT '{QuarterLastWorkingDate}' AS ReferenceDate
                        , l.ExposureKey AS AccountNumber
                        , CAST(`{env_prefix}-cdh`.derivative.converttoeur(l.Currency, l.OutstandingTotalBalance, l.BusinessDate) AS FLOAT) AS OutstandingTotalBalanceEuro
                        , l.AgreementKey AS ContractId
                        FROM Quarterdexposure l
                        INNER JOIN temp1 t
                        ON l.AgreementKey = t.ContractId
                    """)

SecuredPerimeter_Quarter.createOrReplaceTempView("SecuredPerimeter_Quarter")

print(f"Quarterly Secured Perimeter has, {Quarter_Secured_Perimeter.count()} records")
print(f"Quarterly Secured Perimeter has, {SecuredPerimeter_Quarter.count()} records")


## Report Export

In [0]:
df_Collections_mmyy = Collections_mmyy.toPandas()
df_Collections_mmyy_prevmonths = Collections_mmyy_prevmonths.toPandas()
df_Collections_mmyynext_prevmonths = Collections_mmyynext_prevmonths.toPandas()
df_SecuredPerimeter_Quarter = SecuredPerimeter_Quarter.toPandas()

df_Collections_mmyy['PaymentAmount'] = df_Collections_mmyy['PaymentAmount'].apply(lambda x: round(x, 2)).astype(float)
df_Collections_mmyy_prevmonths['PaymentAmount'] = df_Collections_mmyy_prevmonths['PaymentAmount'].apply(lambda x: round(x, 2)).astype(float)
df_Collections_mmyynext_prevmonths['PaymentAmount'] = df_Collections_mmyynext_prevmonths['PaymentAmount'].apply(lambda x: round(x, 2)).astype(float)
df_SecuredPerimeter_Quarter['OutstandingTotalBalanceEuro'] = df_SecuredPerimeter_Quarter['OutstandingTotalBalanceEuro'].apply(lambda x: round(x, 2))

In [0]:
#Debug
'''
print("=== Checking Variable Types ===")
print(f"RefMonth: type={type(RefMonth)}, value={RefMonth}")
print(f"RefMonth_mmyynext: type={type(RefMonth_mmyynext)}, value={RefMonth_mmyynext}")
print(f"QuarterRefMonth: type={type(QuarterRefMonth)}, value={QuarterRefMonth}")
'''

In [0]:
%pip install xlsxwriter

from datetime import datetime

# Set current timestamp string
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%d')

# Define the output path
output_path = f"{report_output}{now_datetime_str}/{output_file_prefix}{eomDate}.xlsx"

# Ensure the output directory exists
dbutils.fs.mkdirs(f"{report_output}{now_datetime_str}")

# Delete the output file if it already exists
if os.path.exists(output_path):
    os.remove(output_path)

# Define the temporary file path
temp_file = f"/tmp/{output_file_prefix}{now_datetime_str}.xlsx"

# Convert RefMonth (string) to datetime.date
RefMonth_date = datetime.strptime(RefMonth, "%Y-%m-%d").date()  # Converts to datetime.date

# Write to Excel with proper sheet names
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='xlsxwriter') as writer:
    # Sheet 1: RefMonth
    df_Collections_mmyy.to_excel(
        writer, 
        sheet_name=f"{RefMonth_date.strftime('%b')} {RefMonth_date.year}", 
        index=False
    )
    
    # Sheet 2: RefMonth for previous months
    df_Collections_mmyy_prevmonths.to_excel(
        writer, 
        sheet_name=f"{RefMonth_date.strftime('%b')} {RefMonth_date.year} (for prev months)", 
        index=False
    )
    
    # Sheet 3: RefMonth_mmyynext (already a date)
    df_Collections_mmyynext_prevmonths.to_excel(
        writer, 
        sheet_name=f"{RefMonth_mmyynext.strftime('%b')} {RefMonth_mmyynext.year} (for prev months)", 
        index=False
    )
    
    # Sheet 4: QuarterRefMonth (already a date)
    df_SecuredPerimeter_Quarter.to_excel(
        writer, 
        sheet_name=f"Secured Perimeter {QuarterRefMonth.strftime('%b')}", 
        index=False
    )

# Save the buffer to the temporary file
with open(temp_file, "wb") as f:
    f.write(output_buffer.getvalue())

# Copy the temporary file to the final output path
shutil.copy(temp_file, output_path)

print(f"Output file is written successfully: {output_path}")


In [0]:
# Print all date variables with descriptive labels
# Delete later debugging 

'''
print("First Date:", FirstDate)
print("Last Date:", LastDate)
print("Last Working Date:", LastWorkingDate)
print("Second Working Date:", SecondWorking)
print("First Working Date Next Month:", FirstWorkingNextMonth)
print("Second Working Date Next Month:", SecondWorkingNextMonth)
print("----------------")
print(f"Reference Month (Next): {RefMonth_mmyynext}")
print(f"First Date of Next Period: {FirstDate_mmyynext}")
print(f"Last Date of Next Period: {LastDate_mmyynext}")
print(f"Last Working Date of Next Period: {LastWorkingDate_mmyynext}")
print(f"Second Working Date of Next Period: {SecondWorking_mmyynext}")
print(f"First Working Date of Following Month: {FirstWorkingNextMonth_mmyynext}")
print(f"Second Working Date of Following Month: {SecondWorkingNextMonth_mmyynext}")
'''